# Первая практика — мини-анализ HMDA

## Подготовка

In [2]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: /opt/anaconda3/bin/python
pandas: 2.2.2
Рабочая папка: /Users/artemposvin/data-analysis-course/lessons/01-introduction/work


## Маленький пример reading_example.csv

In [3]:
example = pd.read_csv(
    DATA / "reading_example.csv",
    dtype={"lei": "string", "county_code": "string"},
)
example

,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,275000.0,72.0
1,3,2023,549300P1347BFCBHJV60,11001,1,585000.0,174.0
2,278,2023,KB1H1DSPRFMYMCUFXT09,11001,2,475000.0,108.0
3,304,2023,549300VZVN841I2ILS84,11001,2,905000.0,353.0
4,21,2023,549300RPOGWJRH63HS39,11001,3,595000.0,184.0
5,27,2023,JJKC32MCHWDI71265Z06,11001,3,1805000.0,667.0


## A. Разделитель CSV

In [4]:
wrong_sep = pd.read_csv(DATA / "reading_example.csv", sep=";")
print(example.shape)
print(wrong_sep.shape)
wrong_sep


(6, 7)
(6, 1)


,"source_row,activity_year,lei,county_code,action_taken,loan_amount,income"
0,"2,2023,549300VZVN841I2ILS84,11001,1,275000.0,72.0"
1,"3,2023,549300P1347BFCBHJV60,11001,1,585000.0,1..."
2,"278,2023,KB1H1DSPRFMYMCUFXT09,11001,2,475000.0..."
3,"304,2023,549300VZVN841I2ILS84,11001,2,905000.0..."
4,"21,2023,549300RPOGWJRH63HS39,11001,3,595000.0,..."
5,"27,2023,JJKC32MCHWDI71265Z06,11001,3,1805000.0..."


## B. Частичное чтение

In [5]:
preview = pd.read_csv(DATA / "reading_example.csv", nrows=3)
small_columns = pd.read_csv(
    DATA / "reading_example.csv", usecols=["action_taken", "income"]
)
print(preview.shape)
print(small_columns.shape)


(3, 7)
(6, 2)


## Выбор Series и DataFrame

In [6]:
outcomes = example["action_taken"]
print(type(outcomes))
outcomes

<class 'pandas.core.series.Series'>


0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: int64

## Выбор нескольких столбцов

In [7]:
names = ["source_row", "action_taken", "income"]
selected = example[names]
print(type(selected))
selected.head(6)

<class 'pandas.core.frame.DataFrame'>


,source_row,action_taken,income
0,2,1,72.0
1,3,1,174.0
2,278,2,108.0
3,304,2,353.0
4,21,3,184.0
5,27,3,667.0


## C. Выбор столбцов и просмотр

In [8]:
amount_view = example[["source_row", "loan_amount"]].head(2)
print(amount_view)
print(amount_view.info())
print(example.shape)
print(example.columns)
example.info()


   source_row  loan_amount
0           2     275000.0
1           3     585000.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   source_row   2 non-null      int64  
 1   loan_amount  2 non-null      float64
dtypes: float64(1), int64(1)
memory usage: 164.0 bytes
None
(6, 7)
Index(['source_row', 'activity_year', 'lei', 'county_code', 'action_taken',
       'loan_amount', 'income'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   source_row     6 non-null      int64  
 1   activity_year  6 non-null      int64  
 2   lei            6 non-null      string 
 3   county_code    6 non-null      string 
 4   action_taken   6 non-null      int64  
 5   loan_amount    6 non-null      float64
 

## Пример расчёта одобрений

In [9]:
approved_flag = outcomes.isin([1, 2])
approved_flag

0     True
1     True
2     True
3     True
4    False
5    False
Name: action_taken, dtype: bool

## Количество и доля одобрений

In [10]:
n_approved = approved_flag.sum()
n_decisions = len(example)
assert n_decisions > 0
small_approval_rate = n_approved / n_decisions
print(n_approved, n_decisions)
print(f"Одобрено: {100 * small_approval_rate:.2f}%")

4 6
Одобрено: 66.67%


## D. Доля выдач на маленьком примере

In [11]:
issued_flag = outcomes.eq(1)
n_issued = issued_flag.sum()
small_origination_rate = n_issued / n_decisions
print(issued_flag)
print(n_issued, n_decisions)
print(f"Выдано: {100 * small_origination_rate:.2f}%")


0     True
1     True
2    False
3    False
4    False
5    False
Name: action_taken, dtype: bool
2 6
Выдано: 33.33%


## Основной набор decisions.csv

In [12]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
full_outcomes = decisions["action_taken"]
full_n = len(decisions)
print(decisions.shape)
full_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## E. Две доли на основном наборе

In [13]:
full_approved = full_outcomes.isin([1, 2]).sum()
full_issued = full_outcomes.eq(1).sum()
full_approval_rate = full_approved / full_n
full_origination_rate = full_issued / full_n
print(f"Одобрено {full_approved}/{full_n}: {100 * full_approval_rate:.2f}%")
print(f"Выдано {full_issued}/{full_n}: {100 * full_origination_rate:.2f}%")


Одобрено 6139/6830: 89.88%
Выдано 5956/6830: 87.20%


## Проверка входных данных

In [14]:
assert full_n > 0
assert full_outcomes.isna().sum() == 0
assert full_outcomes.isin([1, 2, 3]).sum() == full_n
income_missing = decisions["income"].isna().sum()
print("Пропущен доход:", income_missing)

Пропущен доход: 342


## F. Баланс и диапазоны

In [15]:
full_denied = full_outcomes.eq(3).sum()
assert full_approved + full_denied == full_n
assert 0 <= full_issued <= full_approved <= full_n
assert 0 <= full_origination_rate <= full_approval_rate <= 1
print("Отказы:", full_denied)


Отказы: 691


## G. Расчёт из отчёта

In [16]:
approval_rate = full_outcomes.eq(1).sum() / full_n
approval_rate

0.8720351390922401

## G. Исправление условия

In [17]:
correct_approval_rate = full_outcomes.isin([1, 2]).sum() / full_n
print(f"Одобрено: {100 * correct_approval_rate:.2f}%")


Одобрено: 89.88%


## G. Исправление названия

In [18]:
origination_rate = full_outcomes.eq(1).sum() / full_n
print(f"Выдано: {100 * origination_rate:.2f}%")


Выдано: 87.20%
